In [ ]:
import pandas as pd
import numpy as np
import talib
from typing import Dict, List, Optional, Tuple
import warnings

def build_stock_features(df: pd.DataFrame, 
                        price_cols: Dict[str, str] = None,
                        volume_col: str = 'volume',
                        date_col: str = 'date',
                        feature_groups: List[str] = None) -> pd.DataFrame:
    """
    构建A股技术指标特征
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含股票数据的DataFrame，至少包含价格和成交量数据
    price_cols : dict, optional
        价格列的映射，格式为 {'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close'}
        默认为 None，会尝试自动识别
    volume_col : str, default 'volume'
        成交量列名
    date_col : str, default 'date'
        日期列名
    feature_groups : list, optional
        需要构建的特征组，可选值：['overlap', 'momentum', 'volume', 'volatility', 'trend', 'cycle', 'pattern']
        默认为None，构建所有特征组
    
    Returns:
    --------
    pd.DataFrame
        包含技术指标特征的DataFrame
    """
    
    # 复制数据避免修改原始数据
    data = df.copy()
    
    # 自动识别价格列
    if price_cols is None:
        possible_names = {
            'open': ['open', 'Open', 'OPEN', 'o', 'O'],
            'high': ['high', 'High', 'HIGH', 'h', 'H'],
            'low': ['low', 'Low', 'LOW', 'l', 'L'],
            'close': ['close', 'Close', 'CLOSE', 'c', 'C']
        }
        
        price_cols = {}
        for key, possible_values in possible_names.items():
            for col in possible_values:
                if col in data.columns:
                    price_cols[key] = col
                    break
        
        if len(price_cols) != 4:
            raise ValueError("无法找到完整的价格数据列 (open, high, low, close)")
    
    # 获取价格数据
    open_prices = data[price_cols['open']].values.astype(float)
    high_prices = data[price_cols['high']].values.astype(float)
    low_prices = data[price_cols['low']].values.astype(float)
    close_prices = data[price_cols['close']].values.astype(float)
    volume = data[volume_col].values.astype(float) if volume_col in data.columns else np.ones(len(data)) * 1000
    
    # 默认构建所有特征组
    if feature_groups is None:
        feature_groups = ['overlap', 'momentum', 'volume', 'volatility', 'trend', 'cycle', 'pattern']
    
    print(f"开始构建特征，数据长度: {len(data)}")
    print(f"构建特征组: {feature_groups}")
    
    # 构建重叠指标 (Overlap Studies)
    if 'overlap' in feature_groups:
        print("构建重叠指标...")
        # 移动平均线
        data['MA_5'] = talib.SMA(close_prices, timeperiod=5)
        data['MA_10'] = talib.SMA(close_prices, timeperiod=10)
        data['MA_20'] = talib.SMA(close_prices, timeperiod=20)
        data['MA_30'] = talib.SMA(close_prices, timeperiod=30)
        data['MA_60'] = talib.SMA(close_prices, timeperiod=60)
        
        # 指数移动平均线
        data['EMA_5'] = talib.EMA(close_prices, timeperiod=5)
        data['EMA_10'] = talib.EMA(close_prices, timeperiod=10)
        data['EMA_20'] = talib.EMA(close_prices, timeperiod=20)
        data['EMA_30'] = talib.EMA(close_prices, timeperiod=30)
        
        # 布林带
        upper, middle, lower = talib.BBANDS(close_prices, timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)
        data['BB_upper'] = upper
        data['BB_middle'] = middle
        data['BB_lower'] = lower
        data['BB_width'] = (upper - lower) / middle  # 布林带宽度
        data['BB_position'] = (close_prices - lower) / (upper - lower)  # 布林带位置
        
        # 其他重叠指标
        data['HT_TRENDLINE'] = talib.HT_TRENDLINE(close_prices)
        data['KAMA'] = talib.KAMA(close_prices, timeperiod=30)
    
    # 构建动量指标 (Momentum Indicators)
    if 'momentum' in feature_groups:
        print("构建动量指标...")
        # RSI
        data['RSI_6'] = talib.RSI(close_prices, timeperiod=6)
        data['RSI_14'] = talib.RSI(close_prices, timeperiod=14)
        data['RSI_24'] = talib.RSI(close_prices, timeperiod=24)
        
        # MACD
        macd, macd_signal, macd_hist = talib.MACD(close_prices, fastperiod=12, slowperiod=26, signalperiod=9)
        data['MACD'] = macd
        data['MACD_signal'] = macd_signal
        data['MACD_hist'] = macd_hist
        data['MACD_diff'] = macd - macd_signal  # MACD差值
        
        # 随机指标
        slowk, slowd = talib.STOCH(high_prices, low_prices, close_prices, 
                                  fastk_period=5, slowk_period=3, slowk_matype=0,
                                  slowd_period=3, slowd_matype=0)
        data['STOCH_K'] = slowk
        data['STOCH_D'] = slowd
        data['STOCH_diff'] = slowk - slowd
        
        # 随机相对强弱指标
        data['STOCHRSI_fastk'] = talib.STOCHRSI(close_prices, timeperiod=14, fastk_period=5, fastd_period=3, fastd_matype=0)[0]
        data['STOCHRSI_fastd'] = talib.STOCHRSI(close_prices, timeperiod=14, fastk_period=5, fastd_period=3, fastd_matype=0)[1]
        
        # 其他动量指标
        data['MOM'] = talib.MOM(close_prices, timeperiod=10)
        data['ROC'] = talib.ROC(close_prices, timeperiod=10)
        data['ROCR'] = talib.ROCR(close_prices, timeperiod=10)
        data['WILLR'] = talib.WILLR(high_prices, low_prices, close_prices, timeperiod=14)
        data['ULTOSC'] = talib.ULTOSC(high_prices, low_prices, close_prices, timeperiod1=7, timeperiod2=14, timeperiod3=28)
        
    # 构建成交量指标 (Volume Indicators)
    if 'volume' in feature_groups:
        print("构建成交量指标...")
        data['AD'] = talib.AD(high_prices, low_prices, close_prices, volume)
        data['ADOSC'] = talib.ADOSC(high_prices, low_prices, close_prices, volume, fastperiod=3, slowperiod=10)
        data['OBV'] = talib.OBV(close_prices, volume)
        
        # 成交量移动平均
        data['VMA_5'] = talib.SMA(volume, timeperiod=5)
        data['VMA_10'] = talib.SMA(volume, timeperiod=10)
        data['VMA_20'] = talib.SMA(volume, timeperiod=20)
        data['VOLUME_RATIO'] = volume / data['VMA_10']  # 成交量比值
    
    # 构建波动率指标 (Volatility Indicators)
    if 'volatility' in feature_groups:
        print("构建波动率指标...")
        data['ATR'] = talib.ATR(high_prices, low_prices, close_prices, timeperiod=14)
        data['NATR'] = talib.NATR(high_prices, low_prices, close_prices, timeperiod=14)
        data['TRANGE'] = talib.TRANGE(high_prices, low_prices, close_prices)
        
        # 价格波动率
        data['STDDEV'] = talib.STDDEV(close_prices, timeperiod=14, nbdev=1)
        data['VAR'] = talib.VAR(close_prices, timeperiod=14, nbdev=1)
    
    # 构建趋势指标 (Trend Indicators)
    if 'trend' in feature_groups:
        print("构建趋势指标...")
        data['ADX'] = talib.ADX(high_prices, low_prices, close_prices, timeperiod=14)
        data['ADXR'] = talib.ADXR(high_prices, low_prices, close_prices, timeperiod=14)
        data['APO'] = talib.APO(close_prices, fastperiod=12, slowperiod=26, matype=0)
        data['PPO'] = talib.PPO(close_prices, fastperiod=12, slowperiod=26, matype=0)
        data['PLUS_DI'] = talib.PLUS_DI(high_prices, low_prices, close_prices, timeperiod=14)
        data['MINUS_DI'] = talib.MINUS_DI(high_prices, low_prices, close_prices, timeperiod=14)
        data['PLUS_DM'] = talib.PLUS_DM(high_prices, low_prices, timeperiod=14)
        data['MINUS_DM'] = talib.MINUS_DM(high_prices, low_prices, timeperiod=14)
        data['DX'] = talib.DX(high_prices, low_prices, close_prices, timeperiod=14)
    
    # 构建周期指标 (Cycle Indicators)
    if 'cycle' in feature_groups:
        print("构建周期指标...")
        data['HT_DCPERIOD'] = talib.HT_DCPERIOD(close_prices)
        data['HT_DCPHASE'] = talib.HT_DCPHASE(close_prices)
        data['HT_PHASOR_inphase'], data['HT_PHASOR_quadrature'] = talib.HT_PHASOR(close_prices)
        data['HT_SINE_sine'], data['HT_SINE_leadsine'] = talib.HT_SINE(close_prices)
        data['HT_TRENDMODE'] = talib.HT_TRENDMODE(close_prices)
    
    # 构建价格模式指标 (Pattern Recognition)
    if 'pattern' in feature_groups:
        print("构建价格模式指标...")
        # 蜡烛图模式识别
        data['CDL2CROWS'] = talib.CDL2CROWS(open_prices, high_prices, low_prices, close_prices)
        data['CDL3BLACKCROWS'] = talib.CDL3BLACKCROWS(open_prices, high_prices, low_prices, close_prices)
        data['CDL3INSIDE'] = talib.CDL3INSIDE(open_prices, high_prices, low_prices, close_prices)
        data['CDL3LINESTRIKE'] = talib.CDL3LINESTRIKE(open_prices, high_prices, low_prices, close_prices)
        data['CDL3OUTSIDE'] = talib.CDL3OUTSIDE(open_prices, high_prices, low_prices, close_prices)
        data['CDL3STARSINSOUTH'] = talib.CDL3STARSINSOUTH(open_prices, high_prices, low_prices, close_prices)
        data['CDL3WHITESOLDIERS'] = talib.CDL3WHITESOLDIERS(open_prices, high_prices, low_prices, close_prices)
        data['CDLABANDONEDBABY'] = talib.CDLABANDONEDBABY(open_prices, high_prices, low_prices, close_prices, penetration=0.3)
        data['CDLADVANCEBLOCK'] = talib.CDLADVANCEBLOCK(open_prices, high_prices, low_prices, close_prices)
        data['CDLBELTHOLD'] = talib.CDLBELTHOLD(open_prices, high_prices, low_prices, close_prices)
        data['CDLBREAKAWAY'] = talib.CDLBREAKAWAY(open_prices, high_prices, low_prices, close_prices)
        data['CDLCLOSINGMARUBOZU'] = talib.CDLCLOSINGMARUBOZU(open_prices, high_prices, low_prices, close_prices)
        data['CDLCONCEALBABYSWALL'] = talib.CDLCONCEALBABYSWALL(open_prices, high_prices, low_prices, close_prices)
        data['CDLCOUNTERATTACK'] = talib.CDLCOUNTERATTACK(open_prices, high_prices, low_prices, close_prices)
        data['CDLDARKCLOUDCOVER'] = talib.CDLDARKCLOUDCOVER(open_prices, high_prices, low_prices, close_prices, penetration=0.5)
        data['CDLDOJI'] = talib.CDLDOJI(open_prices, high_prices, low_prices, close_prices)
        data['CDLDOJISTAR'] = talib.CDLDOJISTAR(open_prices, high_prices, low_prices, close_prices)
        data['CDLDRAGONFLYDOJI'] = talib.CDLDRAGONFLYDOJI(open_prices, high_prices, low_prices, close_prices)
        data['CDLENGULFING'] = talib.CDLENGULFING(open_prices, high_prices, low_prices, close_prices)
        data['CDLEVENINGDOJISTAR'] = talib.CDLEVENINGDOJISTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3)
        data['CDLEVENINGSTAR'] = talib.CDLEVENINGSTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3)
        data['CDLGAPSIDESIDEWHITE'] = talib.CDLGAPSIDESIDEWHITE(open_prices, high_prices, low_prices, close_prices)
        data['CDLGRAVESTONEDOJI'] = talib.CDLGRAVESTONEDOJI(open_prices, high_prices, low_prices, close_prices)
        data['CDLHAMMER'] = talib.CDLHAMMER(open_prices, high_prices, low_prices, close_prices)
        data['CDLHANGINGMAN'] = talib.CDLHANGINGMAN(open_prices, high_prices, low_prices, close_prices)
        data['CDLHARAMI'] = talib.CDLHARAMI(open_prices, high_prices, low_prices, close_prices)
        data['CDLHARAMICROSS'] = talib.CDLHARAMICROSS(open_prices, high_prices, low_prices, close_prices)
        data['CDLHIGHWAVE'] = talib.CDLHIGHWAVE(open_prices, high_prices, low_prices, close_prices)
        data['CDLHIKKAKE'] = talib.CDLHIKKAKE(open_prices, high_prices, low_prices, close_prices)
        data['CDLHIKKAKEMOD'] = talib.CDLHIKKAKEMOD(open_prices, high_prices, low_prices, close_prices)
        data['CDLHOMINGPIGEON'] = talib.CDLHOMINGPIGEON(open_prices, high_prices, low_prices, close_prices)
        data['CDLIDENTICAL3CROWS'] = talib.CDLIDENTICAL3CROWS(open_prices, high_prices, low_prices, close_prices)
        data['CDLINNECK'] = talib.CDLINNECK(open_prices, high_prices, low_prices, close_prices)
        data['CDLINVERTEDHAMMER'] = talib.CDLINVERTEDHAMMER(open_prices, high_prices, low_prices, close_prices)
        data['CDLKICKING'] = talib.CDLKICKING(open_prices, high_prices, low_prices, close_prices)
        data['CDLKICKINGBYLENGTH'] = talib.CDLKICKINGBYLENGTH(open_prices, high_prices, low_prices, close_prices)
        data['CDLLADDERBOTTOM'] = talib.CDLLADDERBOTTOM(open_prices, high_prices, low_prices, close_prices)
        data['CDLLONGLEGGEDDOJI'] = talib.CDLLONGLEGGEDDOJI(open_prices, high_prices, low_prices, close_prices)
        data['CDLLONGLINE'] = talib.CDLLONGLINE(open_prices, high_prices, low_prices, close_prices)
        data['CDLMARUBOZU'] = talib.CDLMARUBOZU(open_prices, high_prices, low_prices, close_prices)
        data['CDLMATCHINGLOW'] = talib.CDLMATCHINGLOW(open_prices, high_prices, low_prices, close_prices)
        data['CDLMATHOLD'] = talib.CDLMATHOLD(open_prices, high_prices, low_prices, close_prices, penetration=0.5)
        data['CDLMORNINGDOJISTAR'] = talib.CDLMORNINGDOJISTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3)
        data['CDLMORNINGSTAR'] = talib.CDLMORNINGSTAR(open_prices, high_prices, low_prices, close_prices, penetration=0.3)
        data['CDLONNECK'] = talib.CDLONNECK(open_prices, high_prices, low_prices, close_prices)
        data['CDLPIERCING'] = talib.CDLPIERCING(open_prices, high_prices, low_prices, close_prices)
        data['CDLRICKSHAWMAN'] = talib.CDLRICKSHAWMAN(open_prices, high_prices, low_prices, close_prices)
        data['CDLRISEFALL3METHODS'] = talib.CDLRISEFALL3METHODS(open_prices, high_prices, low_prices, close_prices)
        data['CDLSEPARATINGLINES'] = talib.CDLSEPARATINGLINES(open_prices, high_prices, low_prices, close_prices)
        data['CDLSHOOTINGSTAR'] = talib.CDLSHOOTINGSTAR(open_prices, high_prices, low_prices, close_prices)
        data['CDLSHORTLINE'] = talib.CDLSHORTLINE(open_prices, high_prices, low_prices, close_prices)
        data['CDLSPINNINGTOP'] = talib.CDLSPINNINGTOP(open_prices, high_prices, low_prices, close_prices)
        data['CDLSTALLEDPATTERN'] = talib.CDLSTALLEDPATTERN(open_prices, high_prices, low_prices, close_prices)
        data['CDLSTICKSANDWICH'] = talib.CDLSTICKSANDWICH(open_prices, high_prices, low_prices, close_prices)
        data['CDLTAKURI'] = talib.CDLTAKURI(open_prices, high_prices, low_prices, close_prices)
        data['CDLTASUKIGAP'] = talib.CDLTASUKIGAP(open_prices, high_prices, low_prices, close_prices)
        data['CDLTHRUSTING'] = talib.CDLTHRUSTING(open_prices, high_prices, low_prices, close_prices)
        data['CDLTRISTAR'] = talib.CDLTRISTAR(open_prices, high_prices, low_prices, close_prices)
        data['CDLUNIQUE3RIVER'] = talib.CDLUNIQUE3RIVER(open_prices, high_prices, low_prices, close_prices)
        data['CDLUPSIDEGAP2CROWS'] = talib.CDLUPSIDEGAP2CROWS(open_prices, high_prices, low_prices, close_prices)
        data['CDLXSIDEGAP3METHODS'] = talib.CDLXSIDEGAP3METHODS(open_prices, high_prices, low_prices, close_prices)
    
    # 添加自定义特征
    print("构建自定义特征...")
    # 价格相关特征
    data['HIGH_LOW_RATIO'] = high_prices / low_prices
    data['CLOSE_OPEN_RATIO'] = close_prices / open_prices
    data['HIGH_CLOSE_RATIO'] = high_prices / close_prices
    data['LOW_CLOSE_RATIO'] = low_prices / close_prices
    
    # 价格变化率
    data['PRICE_CHANGE'] = (close_prices - np.roll(close_prices, 1)) / np.roll(close_prices, 1)
    data['HIGH_LOW_PCT'] = (high_prices - low_prices) / close_prices
    data['HIGH_CLOSE_PCT'] = (high_prices - close_prices) / close_prices
    data['CLOSE_LOW_PCT'] = (close_prices - low_prices) / close_prices
    
    # 移动平均线之间的关系
    if 'MA_5' in data.columns and 'MA_10' in data.columns:
        data['MA_5_10_RATIO'] = data['MA_5'] / data['MA_10']
        data['MA_5_10_DIFF'] = data['MA_5'] - data['MA_10']
    
    if 'MA_20' in data.columns and 'MA_60' in data.columns:
        data['MA_20_60_RATIO'] = data['MA_20'] / data['MA_60']
        data['MA_20_60_DIFF'] = data['MA_20'] - data['MA_60']
    
    # 波动率相关特征
    if 'STDDEV' in data.columns and 'close' in data.columns:
        data['VOLATILITY_RATIO'] = data['STDDEV'] / data[price_cols['close']]
    
    # 处理缺失值
    print("处理缺失值...")
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.fillna(method='bfill').fillna(method='ffill')
    
    print(f"特征构建完成，共生成 {len(data.columns)} 个特征")
    return data

def add_target_variable(df: pd.DataFrame, 
                       target_col: str = 'target',
                       look_forward: int = 5,
                       threshold: float = 0.02) -> pd.DataFrame:
    """
    添加目标变量（未来收益率分类）
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含特征的数据框
    target_col : str
        目标列名
    look_forward : int
        向前看的天数
    threshold : float
        分类阈值
    
    Returns:
    --------
    pd.DataFrame
        包含目标变量的数据框
    """
    data = df.copy()
    
    # 计算未来收益率
    future_close = np.roll(data['close'].values, -look_forward)
    current_close = data['close'].values
    future_return = (future_close - current_close) / current_close
    
    # 创建分类目标变量
    data[target_col] = 0  # 默认为0（持有或小幅上涨）
    data.loc[future_return > threshold, target_col] = 1  # 显著上涨
    data.loc[future_return < -threshold, target_col] = -1  # 显著下跌
    
    # 最后几行无法计算未来收益率，设为NaN
    data.iloc[-look_forward:, data.columns.get_loc(target_col)] = np.nan
    
    return data

def get_feature_summary(df: pd.DataFrame) -> Dict[str, List[str]]:
    """
    获取特征摘要信息
    
    Parameters:
    -----------
    df : pd.DataFrame
        包含特征的数据框
    
    Returns:
    --------
    dict
        按类型分组的特征列表
    """
    features = df.columns.tolist()
    
    # 特征分类
    overlap_features = [f for f in features if any(x in f.lower() for x in ['ma', 'ema', 'bb', 'ht_', 'kama'])]
    momentum_features = [f for f in features if any(x in f.lower() for x in ['rsi', 'macd', 'stoch', 'mom', 'roc', 'willr', 'ultosc'])]
    volume_features = [f for f in features if any(x in f.lower() for x in ['ad', 'adosc', 'obv', 'vma', 'volume'])]
    volatility_features = [f for f in features if any(x in f.lower() for x in ['atr', 'natr', 'trange', 'stddev', 'var'])]
    trend_features = [f for f in features if any(x in f.lower() for x in ['adx', 'adxr', 'apo', 'ppo', 'di', 'dm', 'dx'])]
    cycle_features = [f for f in features if any(x in f.lower() for x in ['ht_'])]
    pattern_features = [f for f in features if 'cdl' in f.lower()]
    custom_features = [f for f in features if f not in (overlap_features + momentum_features + 
                                                      volume_features + volatility_features + 
                                                      trend_features + cycle_features + pattern_features) 
                      and f not in ['date', 'open', 'high', 'low', 'close', 'volume', 'target']]
    
    summary = {
        'overlap': overlap_features,
        'momentum': momentum_features,
        'volume': volume_features,
        'volatility': volatility_features,
        'trend': trend_features,
        'cycle': cycle_features,
        'pattern': pattern_features,
        'custom': custom_features
    }
    
    return summary

# 使用示例
if __name__ == "__main__":
    # 创建示例数据
    dates = pd.date_range('2023-01-01', periods=1000, freq='D')
    np.random.seed(42)
    
    sample_data = pd.DataFrame({
        'date': dates,
        'open': 100 + np.cumsum(np.random.randn(1000) * 0.5),
        'high': 100 + np.cumsum(np.random.randn(1000) * 0.5) + np.random.rand(1000) * 2,
        'low': 100 + np.cumsum(np.random.randn(1000) * 0.5) - np.random.rand(1000) * 2,
        'close': 100 + np.cumsum(np.random.randn(1000) * 0.5),
        'volume': np.random.randint(1000000, 10000000, 1000)
    })
    
    # 确保high >= low >= close >= open的基本关系
    sample_data['high'] = np.maximum(sample_data['high'], sample_data[['open', 'close']].max(axis=1) + 0.1)
    sample_data['low'] = np.minimum(sample_data['low'], sample_data[['open', 'close']].min(axis=1) - 0.1)
    
    print("示例数据形状:", sample_data.shape)
    print("示例数据前5行:")
    print(sample_data.head())
    
    # 构建特征
    feature_data = build_stock_features(sample_data, 
                                      price_cols={'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close'},
                                      volume_col='volume',
                                      feature_groups=['overlap', 'momentum', 'volume'])
    
    print("\n特征数据形状:", feature_data.shape)
    print("特征列数:", len(feature_data.columns))
    
    # 添加目标变量
    feature_data = add_target_variable(feature_data, look_forward=5, threshold=0.03)
    
    # 获取特征摘要
    feature_summary = get_feature_summary(feature_data)
    
    print("\n特征摘要:")
    for category, features in feature_summary.items():
        if features:
            print(f"{category}: {len(features)} 个特征")
            print(f"  示例: {features[:3] if len(features) > 3 else features}")
    
    print(f"\n最终数据形状: {feature_data.shape}")
    print("特征构建完成!")